# Can we predict next-month account churn?

**Target:** `churn_next_month` (1 = account cancels in the month after the snapshot month).
**Data:** monthly account snapshots (`data.csv`), described in `columns.csv`.

Plan:
1. Audit the data and check every column is legitimately available *at prediction time*.
2. Fit baselines and a gradient-boosted model with leakage-safe validation (grouped by account, plus a forward-in-time holdout).
3. Judge whether the result is useful, not just whether a metric is high.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)

df = pd.read_csv('data.csv')
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| churn rate: %.1f%%' % (100*df.churn_next_month.mean()))
print('missing values:', int(df.isna().sum().sum()), '| duplicate account-months:', int(df.duplicated(['account_ref','snapshot_month']).sum()))
df.describe().T

## 1. Data audit

Each account appears in many monthly rows (about 5 on average), so rows are **not independent**: a random row split would put the same account in train and test. All validation below is grouped by `account_ref`.

### Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from the CRM export of 2026-01-05*. That is measured after the snapshot, so it can encode the future. Compare it with `tenure_months` (months so far):

In [ ]:
gap = df.lifetime_months - df.tenure_months
print(gap.groupby(df.churn_next_month).describe())
print('\nChurners with lifetime == tenure:    %.1f%%' % (100*(gap[df.churn_next_month==1]==0).mean()))
print('Non-churners with lifetime == tenure: %.1f%%' % (100*(gap[df.churn_next_month==0]==0).mean()))

Every churning row has `lifetime_months == tenure_months`, while non-churners almost always have a larger value. The column reveals the outcome (the account stopped being active right after the snapshot) and would not be known for a live account. **It is excluded from all models.** I show below what it would do to the score, to illustrate how misleading it is.

### Churn over time

In [ ]:
m = df.groupby('month').churn_next_month.agg(['mean','size'])
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(m.index.astype(str), m['mean'], marker='o', color='#2a6fdb')
ax.set_ylabel('churn rate'); ax.set_title('Churn rate by snapshot month'); plt.xticks(rotation=70); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
m.T.round(3)

Churn is far from stationary: ~3-5% in spring 2024, then a spike to **40-55% for Nov 2024 - Jan 2025**, back to ~10% from Mar 2025. Whatever drove that (an event, price change, seasonality) is a major source of signal and a major risk: a model that just learns "winter = churn" fails if the event does not repeat. The forward-in-time test below addresses this.

Within the winter spike (Nov 2024 - Jan 2025) vs. other months, the *same* usage/ticket pattern holds, just shifted up. Check:

In [ ]:
w = df.snapshot_month.between('2024-11', '2025-01')
out = pd.concat({'usage quintile': df.groupby([w.rename('winter'), pd.qcut(df.usage_hours, 5)], observed=True).churn_next_month.mean().unstack(0),
                 'tickets': df.groupby([w.rename('winter'), df.support_tickets.clip(upper=4)]).churn_next_month.mean().unstack(0)})
out.round(3)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3))
for ax, c in zip(axs, ['tenure_months','usage_hours','support_tickets','seats']):
    key = pd.qcut(df[c], q=6, duplicates='drop') if df[c].nunique() > 8 else df[c]
    gm = df.groupby(key, observed=True).churn_next_month.mean()
    ax.plot(range(len(gm)), gm.values, marker='o'); ax.set_xticks(range(len(gm)))
    ax.set_xticklabels([f'{i.left:.0f}-{i.right:.0f}' if hasattr(i, 'left') else str(i) for i in gm.index], rotation=45, fontsize=7)
    ax.set_title(c); ax.set_ylim(0, None); ax.grid(alpha=.3)
axs[0].set_ylabel('churn rate'); plt.tight_layout(); plt.show()
df.groupby('plan').agg(rate=('churn_next_month','mean'), fee=('monthly_fee','median'), seats=('seats','median'), n=('plan','size'))

## 2. Features
Only information known at the snapshot: plan, seats, fee, fee per seat, tenure, usage, tickets, and calendar month (kept as a separate feature set so we can see how much it matters). Per-account history (usage change vs. last month, ticket history) uses **earlier** snapshots only.

In [ ]:
df = df.sort_values(['account_ref','month']).reset_index(drop=True)
df['fee_per_seat'] = df.monthly_fee / df.seats
df['cal_month'] = df.month.dt.month
g = df.groupby('account_ref')
df['usage_delta'] = df.usage_hours - g.usage_hours.shift(1)
df['usage_per_seat'] = df.usage_hours / df.seats
df['tickets_prev'] = g.support_tickets.shift(1)
df['tickets_cum'] = g.support_tickets.cumsum()
BASE = ['plan','seats','monthly_fee','fee_per_seat','tenure_months','usage_hours','usage_per_seat','support_tickets']
HIST = ['usage_delta','tickets_prev','tickets_cum']
CAL = ['cal_month']
y = df.churn_next_month.values; groups = df.account_ref.values
print('non-consecutive month gaps within accounts:', int((g.month.diff().dropna().apply(lambda x: x.n) != 1).sum()))

## 3. Models with account-grouped cross-validation
Metrics: ROC-AUC (ranking quality) and PR-AUC (average precision). The base rate is 15%, so a useless model scores ~0.15 PR-AUC.

In [ ]:
def prep(feats):
    X = df[feats].copy()
    if 'plan' in feats: X['plan'] = X.plan.astype('category')
    return X

def make(kind, feats):
    if kind == 'gbm':
        est = HistGradientBoostingClassifier(max_depth=4, learning_rate=.05, max_iter=300, l2_regularization=1.0,
                                             categorical_features=[c == 'plan' for c in feats], random_state=0)
        return est, prep(feats)
    num = [c for c in feats if c != 'plan']
    tf = [('n', StandardScaler(), num)] + ([('c', OneHotEncoder(), ['plan'])] if 'plan' in feats else [])
    # logistic regression cannot take NaN: impute history gaps with 0 (no prior month)
    return make_pipeline(ColumnTransformer(tf), LogisticRegression(max_iter=2000)), df[feats].fillna(0)

gkf = GroupKFold(n_splits=5)
sets = {'A. account attributes only': BASE,
        'B. + history features': BASE + HIST,
        'C. + calendar month': BASE + HIST + CAL}
rows = []
for name, f in sets.items():
    for kind in ['logit', 'gbm']:
        est, X = make(kind, f)
        p = cross_val_predict(est, X, y, groups=groups, cv=gkf, method='predict_proba')[:, 1]
        rows.append((name, kind, roc_auc_score(y, p), average_precision_score(y, p)))
pd.DataFrame(rows, columns=['features','model','ROC-AUC','PR-AUC']).round(3)

### Illustration of the leakage (do not use)

In [ ]:
lk = BASE + ['lifetime_months']
est = HistGradientBoostingClassifier(max_depth=4, learning_rate=.05, max_iter=300, categorical_features=[c == 'plan' for c in lk], random_state=0)
p = cross_val_predict(est, prep(lk), y, groups=groups, cv=gkf, method='predict_proba')[:, 1]
print('GBM WITH lifetime_months: ROC-AUC %.3f  PR-AUC %.3f  <- artefact of leakage' % (roc_auc_score(y, p), average_precision_score(y, p)))

## 4. Forward-in-time holdout
The realistic use is: train on the past, score the next month. Train on snapshots up to 2025-05 and test on 2025-06 to 2025-11 (labels through the 2025-11 snapshot are observable given the 2026-01-05 export). This period contains no churn spike, so it checks whether the signal survives outside the unusual winter. I also report accounts never seen in training.

In [ ]:
cut = pd.Period('2025-06', 'M')
tr = (df.month < cut).values; te = ~tr
new = ~df.account_ref.isin(set(df.account_ref[tr])).values
print('train rows', tr.sum(), '| test rows', te.sum(), '| test churn %.1f%% (train %.1f%%)' % (100*y[te].mean(), 100*y[tr].mean()), '| test rows from unseen accounts', (te & new).sum())
rows = []
for name, f in sets.items():
    for kind in ['logit', 'gbm']:
        est, X = make(kind, f); est.fit(X[tr], y[tr])
        p = est.predict_proba(X[te])[:, 1]; pn = est.predict_proba(X[te & new])[:, 1]
        rows.append((name, kind, roc_auc_score(y[te], p), average_precision_score(y[te], p), roc_auc_score(y[te & new], pn)))
pd.DataFrame(rows, columns=['features','model','ROC-AUC','PR-AUC','ROC-AUC (unseen accts)']).round(3)

## 5. What drives the prediction, and is it usable?
Permutation importance on the forward holdout for set B (attributes + history), plus lift at practical flagging thresholds.

In [ ]:
f = BASE + HIST
est, X = make('gbm', f); est.fit(X[tr], y[tr])
pi = permutation_importance(est, X[te], y[te], scoring='roc_auc', n_repeats=10, random_state=0)
imp = pd.Series(pi.importances_mean, index=f).sort_values()
p = est.predict_proba(X[te])[:, 1]
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
imp.plot.barh(ax=axs[0], color='#2a6fdb'); axs[0].set_title('Permutation importance (drop in ROC-AUC), test period')
pr, rc, _ = precision_recall_curve(y[te], p)
axs[1].plot(rc, pr); axs[1].axhline(y[te].mean(), ls='--', color='gray', label='base rate')
axs[1].set_xlabel('recall'); axs[1].set_ylabel('precision'); axs[1].legend(); axs[1].set_title('Precision-recall, test period')
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(4).to_string())
yt = y[te]; order = np.argsort(-p)
for k in [0.05, 0.1, 0.2]:
    top = order[:int(len(p)*k)]
    print(f'Flag top {k:.0%}: precision {yt[top].mean():.1%} (base {yt.mean():.1%}), captures {yt[top].sum()/yt.sum():.1%} of churn, lift {yt[top].mean()/yt.mean():.1f}x')

## 6. Conclusions

**Short answer: partly. Churn is predictable well above chance, but only modestly at the account level, and the headline numbers depend on how you validate.**

| Setting | ROC-AUC | PR-AUC (base rate) |
|---|---|---|
| Grouped 5-fold CV, account attributes + history | ~0.67-0.68 | ~0.28-0.29 (0.15) |
| Grouped CV, GBM **with calendar month** | **0.80** | 0.48 (0.15) |
| Forward-in-time test (train to 2025-05, test 2025-06+) | ~0.69 | ~0.21 (0.09) |
| Forward test, only accounts unseen in training | ~0.71 | n/a |
| Any model with `lifetime_months` | 1.00 | 1.00 - **leakage, invalid** |

What the evidence says:
- **`lifetime_months` must not be used.** It equals `tenure_months` for 100% of churners and never for non-churners, so it is the label in disguise (it is derived from a 2026-01-05 export, after the outcomes). A perfect score from it is meaningless.
- **Real signal comes from usage and support tickets**: low usage and more tickets mean higher churn, in every period. Plan, seats, fee and tenure add little. Account-level discrimination is moderate (AUC ~0.69).
- **Calendar time is the biggest single effect, and it is not safely transferable.** Churn jumps from ~10% to ~48% for Nov 2024 - Jan 2025 snapshots, uniformly across plan, tenure, usage and tickets. A GBM with the month gets AUC 0.80 in grouped CV purely by learning that spike; on the forward holdout (no spike) the month feature gives no gain and the GBM gets slightly worse (0.67). The 0.80 therefore reflects one historical episode, not a repeatable capability. We do not know what caused it (price change, outage, seasonality, data artefact); this needs a business answer before the model can be trusted to forecast it.
- **Practical value:** in the forward test, flagging the top 5% of account-months by risk gives ~30% precision against a 9% base rate (3.3x lift, ~16% of churn captured); the top 20% gives 2x lift and ~41% capture. That is useful for prioritising outreach, not for confidently identifying who will leave.

Caveats and next steps:
- The forward test covers 6 months with a single regime; confidence intervals are not computed and results should be read to ~±0.02 AUC.
- Rows from the same account are correlated; all CV is grouped by account, but the forward test still includes accounts seen in training (an unseen-accounts result is reported alongside).
- Find out what happened in Nov 2024 - Jan 2025 and whether a leading indicator exists; add richer behavioural data (feature-level usage, billing events, NPS, contract renewal dates) which is likely needed to push beyond AUC ~0.7.
- If deployed, calibrate scores per period (the base rate moved between 3% and 55%) and monitor drift.